Thông tin thêm về data: https://docs.google.com/document/d/1ybmCfRUFrNr0RQHwAHM1PL_QsAqyxeH0yx-oH0yFkNM/edit?usp=sharing
docs lab: https://docs.google.com/document/d/17GA-TDwblfZ6vgF5iXUhn3GWcW1jtiAwd1cl1durNL4/edit?tab=t.0

In [30]:
import pandas as pd
import re
from nltk import ngrams
from gensim.models import Word2Vec
import numpy as np

In [31]:
df = pd.read_csv("/kaggle/input/datasets/phamtheds/news-dataset-vietnameses/Dataset_articles_NoID.csv").dropna()

In [32]:
len(df)

306968

In [33]:
df.head()

,URL,Title,Summary,Contents,Date,Author(s),Category,Tags
0,https://laodong.vn/bat-dong-san/thong-tin-ngoc...,"Thông tin “Ngọc Trinh mua đất ở Bảo Lộc"" chỉ l...","Lâm Đồng - Lãnh đạo thành phố Bảo Lộc, Lâm Đồn...","Những ngày vừa qua, trên trang Facebook chính ...","Thứ sáu, 20/05/2022 08:56 (GMT+7)",Phương Nhiên,Bất động sản,"['Lâm Đồng', 'Ngọc Trinh', 'Chiêu trò', 'Giá đ..."
1,https://laodong.vn/bat-dong-san/lo-hong-trong-...,Lỗ hổng trong việc thẩm tra năng lực tài chính...,TPHCM - Việc không thể cưỡng chế thuế của hai ...,"Theo thông tin từ Cục Thuế TP.HCM, hiện cơ qua...","Thứ sáu, 20/05/2022 08:10 (GMT+7)",Gia Miêu,Bất động sản,"['Thủ Thiêm', 'Đấu giá đất']"
2,https://laodong.vn/bat-dong-san/som-hoan-thien...,Sớm hoàn thiện các dự án nhà ở xã hội để CNLĐ ...,"Hiện trên địa bàn tỉnh Ninh Bình có 32 khu, cụ...",CNLĐ mong muốn sớm được tiếp cận với nhà ở xã ...,"Thứ sáu, 20/05/2022 07:47 (GMT+7)",NGUYỄN TRƯỜNG,Bất động sản,"['Dự án', 'Nhà ở xã hội', 'Dự án nhà ở xã hội'..."
3,https://laodong.vn/bat-dong-san/chi-tiet-ho-so...,Chi tiết hồ sơ hoàn công nhà ở năm 2022,Hoàn công nhà ở với ý nghĩa là điều kiện để đư...,Hoàn công nhà ở là một thủ tục hành chính tron...,"Thứ sáu, 20/05/2022 06:44 (GMT+7)",Kim Nhung (T/H),Bất động sản,"['Giấy phép xây dựng', 'Hồ sơ hoàn công', 'nhà..."
4,https://laodong.vn/bat-dong-san/khoi-tao-khong...,"Khởi tạo không gian sống đẳng cấp, đón sóng đầ...",Có rất nhiều lý do khiến những dự án thấp nội ...,Đi dọc đường Lê Văn Lương kéo dài xuống khu Dư...,"Thứ năm, 19/05/2022 15:30 (GMT+7)",Huyền Nguyễn,Bất động sản,['An Quý Villa']


In [34]:
df['Category'].unique()

array(['Bất động sản', 'Xã hội', 'Lao Động cuối tuần', 'Kinh doanh',
       'Thời sự', 'Công đoàn', 'Bạn đọc', 'Pháp luật',
       'Sự kiện Bình luận', 'Xe +', 'Thế giới', 'Văn hóa - Giải trí',
       'Sổ tay kinh tế', 'Lao Động & Đời sống', 'Phóng sự', 'Lưu trữ',
       'Tin địa phương', 'Tin bài liên quan', 'Giáo dục', 'Sức khỏe',
       'Thể thao', 'Media', 'Video', 'Diễn đàn', 'Tấm Lòng Vàng',
       'Công nghệ', 'Thông tin doanh nghiệp', 'Tin bài nổi bật',
       'Gia đình - Hôn nhân', 'Người Việt tử tế', 'Thông tin tiện ích',
       'Tin tức việc làm', 'Lao Động Xuân', 'Tin hoạt động', 'Du lịch',
       'Tin bài xem thêm', 'Tản mạn - Chuyện dọc đường',
       'Phóng sự - Điều tra', 'Quỹ TLV'], dtype=object)

In [35]:
df['Category'].value_counts().head(15)

Category
Xã hội                 57942
Thể thao               31050
Thế giới               28978
Pháp luật              28703
Kinh doanh             28416
Công đoàn              23461
Giáo dục               22731
Văn hóa - Giải trí     19250
Thời sự                17188
Sức khỏe               12062
Bạn đọc                 8571
Media                   7261
Bất động sản            6722
Gia đình - Hôn nhân     4381
Xe +                    4108
Name: count, dtype: int64

In [36]:
def sample_by_category(group):
    if group.name in ['Công nghệ', 'Giáo dục', 'Sức khỏe', 'Thể thao']:
        return group.sample(n=min(len(group), 300), random_state=42)
    else:
        return group.sample(n=min(len(group), 100), random_state=42)

df_sample = df.groupby('Category', group_keys=False).apply(sample_by_category)

/tmp/ipykernel_55/4256678124.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby('Category', group_keys=False).apply(sample_by_category)


In [37]:
len(df_sample)

3440

In [38]:
labels_1 = ['Công nghệ', 'Giáo dục', 'Sức khỏe', 'Thể thao']

df_sample['label'] = df_sample['Category'].isin(labels_1).astype(int)

In [39]:
df_sample['label'].value_counts()

label
0    2240
1    1200
Name: count, dtype: int64

In [40]:
# Độ dài ký tự của mỗi dòng
df_sample['length'] = df_sample['Contents'].astype(str).apply(len)

# Min, Max
print("Min:", df_sample['length'].min())
print("Max:", df_sample['length'].max())

Min: 17
Max: 25158


In [41]:
df_sample['Summary'].str.len().describe()

count    3440.000000
mean      205.281395
std        83.759924
min        45.000000
25%       144.000000
50%       189.000000
75%       250.000000
max       697.000000
Name: Summary, dtype: float64

In [42]:
df_sample = df_sample.drop(columns = ['Date', 'Author(s)', 'Tags'])

In [43]:
df_sample

,URL,Title,Summary,Contents,Category,label,length
304873,https://laodong.vn/ban-doc/nghe-si-lam-tu-thie...,"Nghệ sĩ làm từ thiện: Người sao kê người chưa,...",Những ồn ào về việc các nghệ sĩ làm từ thiện k...,Ca sĩ Thủy Tiên có lúc phải bật khóc trước sự ...,Bạn đọc,0,2984
306606,https://laodong.vn/ban-doc/so-gddt-tinh-ca-mau...,Sở GDĐT tỉnh Cà Mau chỉ đạo xử lý vụ nữ sinh b...,Nữ sinh lớp 9 tại một trường học thuộc huyện N...,Để chấn chỉnh kịp thời tình trạng bạo lực học ...,Bạn đọc,0,2043
302956,https://laodong.vn/ban-doc/quy-dinh-ve-ma-so-b...,Quy định về mã số bảo hiểm xã hội của mỗi ngườ...,Bản chất của mã bảo hiểm xã hội là mã định dan...,Bạn đọc hỏi: Hiện tại tôi có 2 thẻ bảo hiểm y ...,Bạn đọc,0,2211
304098,https://laodong.vn/ban-doc/cat-toc-0-dong-doi-...,"Cắt tóc 0 đồng, đổi lấy nụ cười",QUẢNG BÌNH - Đại dịch COVID-19 ảnh hưởng rất n...,Hành trình giúp đỡ mọi người Anh Nguyễn Trung ...,Bạn đọc,0,4384
310508,https://laodong.vn/ban-doc/tu-phan-anh-cua-bao...,"Từ phản ánh của Báo Lao Động, Công an bắt đối ...",Sau khi báo Lao Động đăng bài “Giấy khám sức k...,"Sau một thời gian theo dõi, thu thập thông tin...",Bạn đọc,0,1715
...,...,...,...,...,...,...,...
84513,https://laodong.vn/xa-hoi/thong-xe-cau-can-mai...,Thông xe cầu cạn Mai Dịch - Nam Thăng Long vào...,"Theo Ban Quản lý dự án Thăng Long, hiện dự án ...",Dự án xây dựng cầu cạn đoạn Mai Dịch - Nam Thă...,Xã hội,0,1023
80212,https://laodong.vn/xa-hoi/dong-nai-kham-benh-t...,"Đồng Nai: Khám bệnh trực tuyến, bệnh nhân khôn...",Người dân khám bệnh tại Bệnh viện đa khoa tỉnh...,"Ngày 21.1, Bệnh viện Đa khoa Đồng Nai ký kết B...",Xã hội,0,1517
92346,https://laodong.vn/xa-hoi/covid-19-hang-nghin-...,COVID-19: Hàng nghìn công nhân chen chúc nhau ...,Hàng loạt công nhân chen chúc đi vào nhà máy ở...,"Trưa 6.4, ông Đặng Ngọc Dũng – Phó Chủ tịch UB...",Xã hội,0,1520
59314,https://laodong.vn/xa-hoi/quang-ninh-thuong-70...,Quảng Ninh thưởng 700 triệu/giải nhất quốc tế:...,Quảng Ninh - Học sinh đạt giải cao nhất cấp qu...,"Các mức thưởng cụ thể như sau: Đặc biệt, học s...",Xã hội,0,2162


In [44]:
# df_sample.to_csv("viewdata.csv", sep='\t', encoding='utf-8')

# Làm sạch data

In [45]:
!pip install underthesea

In [46]:
import unicodedata
import re
from underthesea import word_tokenize
stopwords = open('/kaggle/input/datasets/thngonquang/vietnamese-stopwords/vietnamese-stopwords-dash.txt', encoding='utf-8').read().splitlines()

In [47]:
def normalize_unicode(text):
    return unicodedata.normalize('NFC', text)

def remove_special_char(text):
    text = re.sub(r'[^0-9a-zA-ZÀ-ỹ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_stopwords(text):
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return " ".join(words)

def tokenize(text):
    return word_tokenize(text, format="text")

In [48]:
def preprocess(text):
    text = normalize_unicode(text)
    text = text.lower()
    text = remove_special_char(text)
    text = tokenize(text)
    text = remove_stopwords(text)
    return text

In [49]:
# X = df_sample['Title'] + ' ' + df_sample['Summary']
# X = df_sample['Summary']
X = df_sample['Contents']

y = df_sample['label']

In [50]:
X = X.apply(preprocess)

In [51]:
X.head()

304873    ca_sĩ thủy_tiên có lúc phải bật khóc trước sự ...
306606    để chấn_chỉnh kịp_thời tình_trạng bạo_lực học_...
302956    bạn_đọc hỏi hiện_tại tôi có 2 thẻ bảo_hiểm_y_t...
304098    hành_trình giúp_đỡ mọi người anh nguyễn_trung_...
310508    sau một thời_gian theo_dõi thu_thập thông_tin ...
Name: Contents, dtype: object

In [52]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

In [53]:
print(len(X_train))
print(len(X_test))


2924
516


In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix 

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# **TFIDF**

In [55]:
tfidf = TfidfVectorizer()
X_train_vectorizer_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_vectorizer = tfidf.transform(X_test).toarray()

## **LogisticRegression**

In [56]:
# model
model = LogisticRegression()
model.fit(X_train_vectorizer_tfidf, y_train)

# Predict
y_pred_tfidf_lr = model.predict(X_test_vectorizer)

accuracy = accuracy_score(y_test, y_pred_tfidf_lr)
print(f"Độ chính xác của mô hình: {accuracy}")

tfidf_lr_report = classification_report(y_test, y_pred_tfidf_lr, output_dict=True)
print(classification_report(y_test, y_pred_tfidf_lr))
print(confusion_matrix(y_test, y_pred_tfidf_lr))

Độ chính xác của mô hình: 0.877906976744186
              precision    recall  f1-score   support

           0       0.85      0.97      0.91       322
           1       0.93      0.73      0.82       194

    accuracy                           0.88       516
   macro avg       0.89      0.85      0.86       516
weighted avg       0.88      0.88      0.87       516

[[312  10]
 [ 53 141]]


## **SVM**

In [57]:
# model
model = LinearSVC()
model.fit(X_train_vectorizer_tfidf, y_train)

# Predict
y_pred_tfidf_svm = model.predict(X_test_vectorizer)

accuracy = accuracy_score(y_test, y_pred_tfidf_svm)
print(f"Độ chính xác của mô hình: {accuracy}")

tfidf_svm_report = classification_report(y_test, y_pred_tfidf_svm, output_dict=True)
print(classification_report(y_test, y_pred_tfidf_svm))
print(confusion_matrix(y_test, y_pred_tfidf_svm))

Độ chính xác của mô hình: 0.9127906976744186
              precision    recall  f1-score   support

           0       0.90      0.96      0.93       322
           1       0.93      0.83      0.88       194

    accuracy                           0.91       516
   macro avg       0.92      0.90      0.90       516
weighted avg       0.91      0.91      0.91       516

[[310  12]
 [ 33 161]]


# WORD2VEC

In [58]:
sentences = [sen.split() for sen in X_train]

model = Word2Vec(
    sentences,
    vector_size=300,
    window=7,
    min_count=2,
    workers=4,
    sg=1,
    epochs=20
)

In [59]:

def document_vector(text, model):
    words = text.split()
    word_vectors = [model.wv[word] for word in words if word in model.wv]
    
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)
    
    return np.mean(word_vectors, axis=0)

X_w2v = np.array([document_vector(text, model) for text in X_train])
X_w2v_test= np.array([document_vector(text, model) for text in X_test])

In [60]:
# model Logistic Regression
model = LogisticRegression()
model.fit(X_w2v, y_train)

# Predict
y_pred_w2v_lr = model.predict(X_w2v_test)

accuracy = accuracy_score(y_test, y_pred_w2v_lr)
print(f"Độ chính xác của mô hình: {accuracy}")

w2v_rb_report = classification_report(y_test, y_pred_w2v_lr, output_dict = True)
print(classification_report(y_test, y_pred_w2v_lr))
print(confusion_matrix(y_test, y_pred_w2v_lr))

Độ chính xác của mô hình: 0.8624031007751938
              precision    recall  f1-score   support

           0       0.85      0.94      0.90       322
           1       0.89      0.73      0.80       194

    accuracy                           0.86       516
   macro avg       0.87      0.84      0.85       516
weighted avg       0.86      0.86      0.86       516

[[304  18]
 [ 53 141]]


In [61]:
# model SVM
model = LinearSVC()
model.fit(X_w2v, y_train)

# Predict
y_pred_w2v_svm = model.predict(X_w2v_test)

accuracy = accuracy_score(y_test, y_pred_w2v_svm)
print(f"Độ chính xác của mô hình: {accuracy}")

w2v_svm_report = classification_report(y_test, y_pred_w2v_svm, output_dict = True)
print(classification_report(y_test, y_pred_w2v_svm))
print(confusion_matrix(y_test, y_pred_w2v_svm))

Độ chính xác của mô hình: 0.875968992248062
              precision    recall  f1-score   support

           0       0.86      0.95      0.91       322
           1       0.90      0.75      0.82       194

    accuracy                           0.88       516
   macro avg       0.88      0.85      0.86       516
weighted avg       0.88      0.88      0.87       516

[[306  16]
 [ 48 146]]


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

results = {
    'TF-IDF + LR':  tfidf_lr_report,
    'TF-IDF + SVM': tfidf_svm_report,
    'W2V + LR':     w2v_rb_report,
    'W2V + SVM':    w2v_svm_report,
}

rows = []
for model_name, report in results.items():
    rows.append({
        'Model': model_name,
        'Accuracy': round(report['accuracy'], 4),
        'Precision (macro)': round(report['macro avg']['precision'], 4),
        'Recall (macro)': round(report['macro avg']['recall'], 4),
        'F1 (macro)': round(report['macro avg']['f1-score'], 4),
    })

df_results = pd.DataFrame(rows).set_index('Model')
print(df_results.to_string())

# Plot
ax = df_results.plot(kind='bar', figsize=(10, 5), ylim=(0.5, 1.0), rot=15)
ax.set_title('So sánh các mô hình')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Pre-trained WORD2VEC

In [ ]:
from gensim.models import KeyedVectors

model_pre = KeyedVectors.load_word2vec_format(
    '/kaggle/input/models/thngonquang/vietnamese-w2v/scikitlearn/default/1/wiki.vi.model.bin',
    binary=True
)
def document_vector(text, model):
    words = text.split()
    word_vectors = [model[w] for w in words if w in model]

    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(word_vectors, axis=0)

In [ ]:
X_w2v_pre = np.array([document_vector(text, model_pre) for text in X_train])
X_w2v_test_pre= np.array([document_vector(text, model_pre) for text in X_test])

In [ ]:
# model
model = LinearSVC()
model.fit(X_w2v_pre, y_train)

# Predict
y_pred = model.predict(X_w2v_test_pre)

accuracy = accuracy_score(y_test, y_pred)
print(f"Độ chính xác của mô hình: {accuracy}")

nb_report = classification_report(y_test, y_pred, output_dict = True)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))